### 실습 1. 라이브러리 준비하기

In [23]:
# 데이터 계산 및 처리를 위한 라이브러리
import numpy as np
import pandas as pd

# 도서 제목을 TF-IDF 벡터로 변환
from sklearn.feature_extraction.text import TfidfVectorizer

# 벡터 사이의 코사인 유사도 계산
from sklearn.metrics.pairwise import cosine_similarity

### 실습 2. 데이터 불러오기

In [24]:
# 불러올 CSV 파일 경로
DATA_PATH = "book_bestseller_clean.csv"

# 도서 데이터 불러오기
df_books = pd.read_csv(
    DATA_PATH,
    encoding="utf-8-sig",
)

# 데이터의 기본 정보 확인
print("데이터 크기:", df_books.shape)
print("컬럼:", df_books.columns.tolist())
print("상품명 결측치:", df_books["상품명"].isna().sum())

# 상품명 앞부분 10개 확인
df_books[["상품명"]].head(10)

데이터 크기: (989, 8)
컬럼: ['순위', '판매상품ID', '상품명', '판매가', '저자', '출판사', '발행일', '분야']
상품명 결측치: 0


,상품명
0,"세네카, 오늘을 빼앗기고 있는 당신에게"
1,홍정기의 장수근육 혁명
2,싯다르타
3,머니 트렌드 2027
4,흔한남매 23
5,2026 해커스 투자자산운용사 실전동형모의고사 10회분+리얼 기출족보 3종
6,시대예보: 수고인류의 시간
7,리더는 언제 차이를 만들어내는가
8,판매의 법칙
9,니체의 초월자


### 실행 결과 해석

- 원본 데이터는 `989행 × 8열`로 구성되어 있다.
- 추천에 사용할 주요 컬럼은 `상품명`이며, 결과 확인에는 `저자`, `출판사`, `분야`도 활용할 수 있다.
- 상품명 결측치가 `0개`이므로 제목이 없는 도서는 없다.
- 데이터가 정상적으로 불러와졌으며 상품명도 한글 깨짐 없이 출력되었다.

### 실습 3. 추천용 데이터 준비하기

In [25]:
# 원본 데이터를 보존하기 위해 복사본 생성
df_reco = df_books.copy()

# 상품명의 결측치, 자료형, 앞뒤 공백 정리
df_reco["상품명"] = (
    df_reco["상품명"]
    .fillna("")
    .astype(str)
    .str.strip()
)

# 상품명이 비어 있는 행을 제외하고 인덱스 재설정
df_reco = (
    df_reco[df_reco["상품명"] != ""]
    .reset_index(drop=True)
)

# 추천에 사용할 데이터 확인
print("추천에 사용할 도서 수:", len(df_reco))
print(
    "인덱스 범위:",
    df_reco.index.min(),
    "~",
    df_reco.index.max(),
)

추천에 사용할 도서 수: 989
인덱스 범위: 0 ~ 988


### 실행 결과 해석

- 원본 데이터의 상품명을 문자열로 변환하고 앞뒤 공백을 제거했다.
- 빈 상품명을 제외한 결과, 추천에 사용할 도서는 `989권`이다.
- 인덱스를 다시 설정하여 범위가 `0~988`로 연속되었다.
- 이후 DataFrame의 인덱스와 TF-IDF 행렬의 행 번호가 같은 도서를 가리키게 된다.

### 실습 4. 콘텐츠 기반 추천 

### 4. 콘텐츠 기반 추천 이해하기

콘텐츠 기반 추천은 선택한 항목의 특징과 비슷한 특징을 가진 항목을 추천하는 방식이다.

이번 실습에서는 도서의 **상품명(제목)**을 특징으로 사용한다.

도서 제목을 TF-IDF 벡터로 변환한 뒤, 코사인 유사도를 이용하여 다른 도서 제목과의 유사성을 비교한다.

```text
도서 제목
→ TF-IDF 벡터 변환
→ 코사인 유사도 계산
→ 유사한 도서 추천

### 실습 5. 코사인 유사도 확인하기

### 5. 코사인 유사도 이해하기

코사인 유사도는 두 벡터의 **방향이 얼마나 비슷한지** 비교하는 방법이다.

- 1에 가까울수록 두 벡터가 매우 비슷하다.
- 0에 가까울수록 공통된 특징이 적다.
- 자기 자신과 비교한 유사도는 일반적으로 1이다.

In [26]:
# 코사인 유사도를 확인할 예시 벡터 생성
vectors = np.array([
    [1, 1],
    [2, 2],
    [1, 0],
])

# 각 벡터 사이의 코사인 유사도 계산
cosine_similarity(vectors)

array([[1.        , 1.        , 0.70710678],
       [1.        , 1.        , 0.70710678],
       [0.70710678, 0.70710678, 1.        ]])

### 실행 결과 해석

`[1, 1]`과 `[2, 2]`는 숫자의 크기는 다르지만 같은 방향을 향하므로 코사인 유사도가 `1`이다.

`[1, 1]`과 `[1, 0]`은 방향이 완전히 같지 않으므로 코사인 유사도가 약 `0.7071`로 계산되었다.

자기 자신과 비교한 유사도는 `1`이므로 실제 도서 추천 결과에서는 선택한 도서 자기 자신을 제외해야 한다.

### 실습 6. 실제 도서 제목을 TF-IDF로 변환하기

### 6. 도서 제목을 TF-IDF로 변환하기

도서 제목을 숫자로 비교하기 위해 TF-IDF 벡터로 변환한다.

- 행: 도서
- 열: 도서 제목에 등장한 단어
- 값: 각 단어의 TF-IDF 가중치

In [27]:
# 추천에 사용할 도서 제목 가져오기
titles = df_reco["상품명"]

# TF-IDF 변환기 생성
tfidf = TfidfVectorizer()

# 도서 제목을 TF-IDF 벡터로 변환
tfidf_matrix = tfidf.fit_transform(titles)

# 변환 결과 확인
print("도서 수:", tfidf_matrix.shape[0])
print("단어 수:", tfidf_matrix.shape[1])
print("TF-IDF 행렬 크기:", tfidf_matrix.shape)

도서 수: 989
단어 수: 2192
TF-IDF 행렬 크기: (989, 2192)


### 실행 결과 해석

- 도서 제목 `989개`가 TF-IDF 벡터로 변환되었다.
- 전체 제목에서 `2,192개`의 단어 특징이 추출되었다.
- TF-IDF 행렬의 크기는 `(989, 2192)`이다.
- 행 하나는 도서 한 권, 열 하나는 제목에서 추출된 단어 하나를 의미한다.
- 각 셀에는 해당 도서 제목에서 단어가 가지는 TF-IDF 가중치가 저장된다.

### 실습 7. 기준 도서 선택하기

### 7. 기준 도서 선택하기

추천의 기준이 될 도서 한 권을 인덱스로 선택한다.

인덱스 번호만 확인하지 않고 해당 인덱스의 실제 상품명도 함께 확인한다.

In [28]:
# 추천 기준으로 사용할 도서의 인덱스
selected_index = 0

# 선택한 인덱스에 해당하는 도서 제목 가져오기
selected_title = df_reco.loc[
    selected_index,
    "상품명",
]

# 선택한 도서 확인
print("선택 도서 인덱스:", selected_index)
print("선택 도서:", selected_title)

선택 도서 인덱스: 0
선택 도서: 세네카, 오늘을 빼앗기고 있는 당신에게


### 실행 결과 해석

- 인덱스 `0`번 도서를 추천 기준 도서로 선택했다.
- 선택된 도서는 `세네카, 오늘을 빼앗기고 있는 당신에게`이다.
- 앞으로 이 도서와 나머지 `988권`의 제목 유사도를 비교한다.

### 실습 8. 선택 도서와 전체 도서의 유사도 계산하기

### 8. 선택 도서와 전체 도서의 유사도 계산하기

선택한 도서의 TF-IDF 벡터를 가져온 뒤, 전체 도서의 TF-IDF 벡터와 코사인 유사도를 계산한다.

계산된 유사도 점수의 개수는 전체 도서 수와 같아야 한다.

In [29]:
# 선택한 도서의 TF-IDF 벡터 가져오기
selected_vector = tfidf_matrix[selected_index]

# 선택한 도서와 전체 도서의 코사인 유사도 계산
similarity_scores = cosine_similarity(
    selected_vector,
    tfidf_matrix,
).flatten()

# 유사도 계산 결과 확인
print("유사도 개수:", len(similarity_scores))
print("전체 도서 수:", len(df_reco))
print("자기 자신과의 유사도:", similarity_scores[selected_index])

유사도 개수: 989
전체 도서 수: 989
자기 자신과의 유사도: 1.0000000000000002


### 실행 결과 해석

- 전체 도서 수와 유사도 점수의 개수가 모두 `989개`이므로, 각 도서마다 유사도 점수가 하나씩 정상적으로 계산되었다.
- `similarity_scores[0]`은 `df_reco.iloc[0]` 도서의 유사도와 대응한다.
- 선택한 도서와 자기 자신을 비교한 유사도는 약 `1.0`으로 가장 높게 계산되었다.
- 출력된 `1.0000000000000002`는 부동소수점 계산 오차이며 실제 의미는 `1.0`과 같다.
- 따라서 추천 결과를 만들 때는 선택한 도서 자기 자신을 반드시 제외해야 한다.

선택 도서 벡터 1개
→ 전체 도서 벡터 989개와 비교
→ 도서마다 유사도 점수 1개 생성
→ 자기 자신은 유사도가 약 1이므로 추천 결과에서 제외

### 실습 9. 유사도가 높은 순서 확인하기

### 9. 유사도가 높은 도서 확인하기

각 도서의 제목과 유사도 점수를 연결한 데이터프레임을 만든다.

유사도를 내림차순으로 정렬하여 선택한 도서와 제목 표현이 비슷한 도서를 확인한다.

In [30]:
# 도서 인덱스, 상품명, 유사도 점수를 데이터프레임으로 생성
score_df = pd.DataFrame({
    "index": np.arange(len(df_reco)),
    "상품명": df_reco["상품명"],
    "similarity": similarity_scores,
})

# 유사도가 높은 순서로 상위 10개 확인
top10_scores = score_df.sort_values(
    "similarity",
    ascending=False,
).head(10)

# 결과 출력
top10_scores

,index,상품명,similarity
0,0,"세네카, 오늘을 빼앗기고 있는 당신에게",1.000000
709,709,돌이킬 수 있는,0.263016
288,288,비전공자도 이해할 수 있는 LLM 수업,0.166661
335,335,품격 있는 대화를 위한 지식 브리핑,0.153552
3,3,머니 트렌드 2027,0.000000
4,4,흔한남매 23,0.000000
5,5,2026 해커스 투자자산운용사 실전동형모의고사 10회분+리얼 기출족보 3종,0.000000
6,6,시대예보: 수고인류의 시간,0.000000
7,7,리더는 언제 차이를 만들어내는가,0.000000
8,8,판매의 법칙,0.000000


### 실행 결과 해석

- 선택한 도서 자기 자신이 유사도 약 `1.0`으로 가장 위에 나타났다.
- `돌이킬 수 있는`은 유사도 약 `0.2630`으로 두 번째로 높았다.
- `비전공자도 이해할 수 있는 LLM 수업`과 `품격 있는 대화를 위한 지식 브리핑`도 일부 공통 단어 때문에 유사도가 계산되었다.
- 추천 후보 제목에 공통으로 포함된 `있는`과 같은 표현이 유사도에 영향을 준 것으로 보인다.
- 나머지 도서의 유사도는 `0`으로, 선택한 도서 제목과 TF-IDF 기준의 공통 단어가 없다는 의미이다.
- 제목의 의미나 도서 분야가 아닌 단어의 일치 정도를 비교하므로, 실제 내용과 관련 없는 도서가 추천될 수 있다.
- 선택한 도서 자기 자신은 추천 결과에 포함하면 안 되므로 다음 단계에서 제외해야 한다.

### 핵심
유사도 높음
≠ 도서 내용이 반드시 비슷함

제목에 같은 단어가 있음
→ TF-IDF 벡터가 일부 겹침
→ 유사도 점수가 발생함

### 실습 10. 자기 자신을 제외하고 Top 5 만들기

유사도 점수를 내림차순으로 정렬한 뒤, 선택한 도서 자기 자신을 제외한다.

남은 도서 중 유사도가 높은 상위 5권을 추천 결과로 선택한다.

In [31]:
# 유사도가 높은 순서로 도서 인덱스 정렬
sorted_indices = similarity_scores.argsort()[::-1]

# 선택한 도서 자기 자신을 제외하고 상위 5개 인덱스 선택
recommended_indices = [
    idx
    for idx in sorted_indices
    if idx != selected_index
][:5]

# 선택된 추천 도서 인덱스 확인
print("추천 도서 인덱스:", recommended_indices)

# 추천 도서의 상품명 가져오기
result = df_reco.loc[
    recommended_indices,
    ["상품명"],
].copy()

# 각 추천 도서의 유사도 점수 추가
result["similarity"] = [
    round(float(similarity_scores[idx]), 4)
    for idx in recommended_indices
]

# 추천 결과 확인
result

추천 도서 인덱스: [np.int64(709), np.int64(288), np.int64(335), np.int64(985), np.int64(984)]


,상품명,similarity
709,돌이킬 수 있는,0.2630
288,비전공자도 이해할 수 있는 LLM 수업,0.1667
335,품격 있는 대화를 위한 지식 브리핑,0.1536
985,2027 공단기 심슨 독해,0.0000
984,책이라면 팔 만큼 1,0.0000


### 실행 결과 해석

- 선택한 도서 자기 자신인 인덱스 `0`은 추천 결과에서 제외되었다.
- 유사도가 높은 순서로 최대 5권의 도서가 선택되었다.
- `돌이킬 수 있는`이 유사도 `0.2630`으로 가장 높은 추천 도서이다.
- `비전공자도 이해할 수 있는 LLM 수업`은 `0.1667`, `품격 있는 대화를 위한 지식 브리핑`은 `0.1536`으로 계산되었다.
- 앞의 세 권은 선택한 도서와 일부 공통 단어가 있어 유사도 점수가 생성된 것으로 보인다.
- 양수인 유사도를 가진 도서가 세 권뿐이어서, 나머지 두 자리에는 유사도가 `0`인 도서가 포함되었다.
- 유사도 `0`은 선택한 도서 제목과 현재 TF-IDF 기준으로 겹치는 단어가 없다는 의미이므로 실제 추천 후보로서의 의미는 낮다.

### 핵심
유사도 내림차순 정렬
→ 자기 자신 제외
→ 앞에서부터 5개 선택
→ Top 5 추천 결과 생성

자기 자신 제외
+ 동일 제목 제외
+ 유사도가 0보다 큰 도서만 추천

### 실습 11. 추천 결과에 메타데이터 추가하기

추천 결과에 상품명뿐만 아니라 저자, 출판사, 분야를 함께 표시한다.

추가 정보를 통해 추천 도서가 선택한 도서와 실제로 관련 있는지 확인할 수 있다.

In [32]:
# 현재 데이터에 존재하는 추천 결과용 컬럼 선택
available_columns = [
    column
    for column in [
        "상품명",
        "저자",
        "출판사",
        "분야",
    ]
    if column in df_reco.columns
]

# 추천 도서의 상품명과 메타데이터 가져오기
result = df_reco.loc[
    recommended_indices,
    available_columns,
].copy()

# 추천 도서별 유사도 점수 추가
result["similarity"] = [
    round(float(similarity_scores[idx]), 4)
    for idx in recommended_indices
]

# 선택한 도서 확인
print("선택 도서:")
print(selected_title)

# 메타데이터가 포함된 추천 결과 확인
print("\n추천 도서:")
display(result)

선택 도서:
세네카, 오늘을 빼앗기고 있는 당신에게

추천 도서:


,상품명,저자,출판사,분야,similarity
709,돌이킬 수 있는,문목하,아작,소설,0.2630
288,비전공자도 이해할 수 있는 LLM 수업,박상길,비즈니스북스,과학,0.1667
335,품격 있는 대화를 위한 지식 브리핑,김진,북플레저,인문,0.1536
985,2027 공단기 심슨 독해,심우철,에스티유니타스,취업/수험서,0.0000
984,책이라면 팔 만큼 1,코지마 아오,대원씨아이,만화,0.0000


### 실행 결과 해석

- 추천 결과에 상품명뿐만 아니라 저자, 출판사, 분야가 정상적으로 추가되었다.
- 유사도가 가장 높은 `돌이킬 수 있는`은 소설 분야이고, 두 번째 도서는 과학 분야이다.
- `품격 있는 대화를 위한 지식 브리핑`은 인문 분야지만, 나머지 추천 도서는 분야가 서로 다르다.
- 유사도 `0`인 도서에는 취업·수험서와 만화 분야의 도서도 포함되었다.
- 현재 추천은 저자, 출판사, 분야를 유사도 계산에 사용하지 않고 상품명만 사용한다.
- 따라서 제목에 공통 단어가 있으면 분야가 달라도 추천될 수 있다.
- 메타데이터는 현재 추천 결과를 검토하기 위한 용도로만 사용되었다.

### 핵심
유사도 계산 기준
→ 상품명만 사용

저자·출판사·분야
→ 유사도 계산에는 사용하지 않음
→ 추천 결과가 적절한지 사람이 검토할 때 사용

### 실습 12. 추천 함수 만들기

지금까지 작성한 추천 과정을 하나의 함수로 묶는다.

추천 함수는 선택한 도서와 전체 도서의 유사도를 계산하고, 자기 자신과 동일한 제목을 제외한 뒤 유사도가 높은 Top N을 반환한다.

In [33]:
# 선택한 도서와 유사한 도서를 추천하는 함수 정의
def recommend_books(
    selected_index,
    df,
    tfidf_matrix,
    top_n=5,
):
    # 전달받은 인덱스가 유효한지 확인
    if selected_index not in df.index:
        raise IndexError(
            f"유효하지 않은 index입니다: {selected_index}"
        )

    # 선택한 도서 제목 가져오기
    selected_title = df.loc[
        selected_index,
        "상품명",
    ]

    # 선택한 도서와 전체 도서의 코사인 유사도 계산
    similarity_scores = cosine_similarity(
        tfidf_matrix[selected_index],
        tfidf_matrix,
    ).flatten()

    # 유사도가 높은 순서로 인덱스 정렬
    sorted_indices = similarity_scores.argsort()[::-1]

    # 추천 도서의 인덱스를 저장할 빈 리스트
    recommended_indices = []

    # 유사도가 높은 도서부터 하나씩 확인
    for idx in sorted_indices:
        # 선택한 도서 자기 자신 제외
        if idx == selected_index:
            continue

        # 선택한 도서와 제목이 동일한 도서 제외
        if df.loc[idx, "상품명"] == selected_title:
            continue

        # 추천 도서 인덱스 추가
        recommended_indices.append(idx)

        # 원하는 추천 개수를 채우면 반복 종료
        if len(recommended_indices) >= top_n:
            break

    # 현재 데이터에 존재하는 결과용 컬럼 선택
    columns = [
        column
        for column in [
            "상품명",
            "저자",
            "출판사",
            "분야",
        ]
        if column in df.columns
    ]

    # 추천 도서의 정보 가져오기
    result = df.loc[
        recommended_indices,
        columns,
    ].copy()

    # 추천 도서별 유사도 점수 추가
    result["similarity"] = [
        round(float(similarity_scores[idx]), 4)
        for idx in recommended_indices
    ]

    # 인덱스를 재설정한 추천 결과 반환
    return result.reset_index(drop=True)

### 실습 13. 추천 함수 실행하기

추천 기준 도서의 인덱스와 Top N 값을 함수에 전달하여 추천 결과를 생성한다.

실행 결과에서 자기 자신이 제외되었는지, 최대 5권이 반환되었는지, 유사도 내림차순으로 정렬되었는지 확인한다.

In [34]:
# 추천 기준으로 사용할 도서 인덱스
selected_index = 0

# 선택한 도서 제목 확인
print(
    "선택 도서:",
    df_reco.loc[selected_index, "상품명"],
)

# 추천 함수를 실행하여 유사한 도서 5권 생성
recommendations = recommend_books(
    selected_index=selected_index,
    df=df_reco,
    tfidf_matrix=tfidf_matrix,
    top_n=5,
)

# 추천 결과 확인
recommendations

선택 도서: 세네카, 오늘을 빼앗기고 있는 당신에게


,상품명,저자,출판사,분야,similarity
0,돌이킬 수 있는,문목하,아작,소설,0.2630
1,비전공자도 이해할 수 있는 LLM 수업,박상길,비즈니스북스,과학,0.1667
2,품격 있는 대화를 위한 지식 브리핑,김진,북플레저,인문,0.1536
3,2027 공단기 심슨 독해,심우철,에스티유니타스,취업/수험서,0.0000
4,책이라면 팔 만큼 1,코지마 아오,대원씨아이,만화,0.0000


In [35]:
# DataFrame의 도서 수와 TF-IDF 행 수가 같은지 확인
assert df_reco.shape[0] == tfidf_matrix.shape[0]

# 추천 결과가 최대 5개인지 확인
assert len(recommendations) <= 5

# 선택한 도서 자기 자신이 추천 결과에서 제외되었는지 확인
assert (
    df_reco.loc[selected_index, "상품명"]
    not in recommendations["상품명"].values
)

print("추천 결과 기본 검증 완료")

추천 결과 기본 검증 완료


### 실행 결과 해석

- 추천 함수가 선택 도서의 인덱스를 전달받아 유사 도서 5권을 정상적으로 반환했다.
- 선택한 도서인 `세네카, 오늘을 빼앗기고 있는 당신에게`는 추천 결과에서 제외되었다.
- 추천 결과는 유사도가 높은 순서로 정렬되었다.
- 함수 안에서 결과 인덱스를 재설정했기 때문에 추천 결과의 인덱스가 `0~4`로 표시되었다.
- 추천 결과에는 상품명, 저자, 출판사, 분야, 유사도 정보가 포함되었다.
- DataFrame의 도서 수와 TF-IDF 행 수가 같고, 추천 개수가 최대 5권이며, 자기 자신이 제외되었다는 기본 조건을 모두 통과했다.
- 유사도 `0`인 도서도 포함되므로 제목에 공통 단어가 있는 추천 후보가 5권보다 적다는 점을 확인할 수 있다.

### 핵심

선택 도서 인덱스 전달

→ 전체 도서와 코사인 유사도 계산

→ 유사도 내림차순 정렬

→ 자기 자신과 동일 제목 제외

→ Top 5 추천 결과 반환

→ 기본 조건 검증 완료

### 실습 14. 추천 결과 저장하기

추천 함수로 생성한 결과를 CSV 파일로 저장한다.

저장한 파일을 다시 불러와 데이터가 정상적으로 보존되었는지 확인한다.

In [36]:
# 추천 결과를 저장할 파일명
OUTPUT_PATH = "chapter05_recommendations.csv"

# 추천 결과를 CSV 파일로 저장
recommendations.to_csv(
    OUTPUT_PATH,
    index=False,
    encoding="utf-8-sig",
)

# 저장한 CSV 파일 다시 불러오기
saved_recommendations = pd.read_csv(
    OUTPUT_PATH,
    encoding="utf-8-sig",
)

# 저장된 파일의 기본 정보 확인
print("저장된 데이터 크기:", saved_recommendations.shape)
print("저장된 컬럼:", saved_recommendations.columns.tolist())

# 저장된 추천 결과 확인
saved_recommendations

저장된 데이터 크기: (5, 5)
저장된 컬럼: ['상품명', '저자', '출판사', '분야', 'similarity']


,상품명,저자,출판사,분야,similarity
0,돌이킬 수 있는,문목하,아작,소설,0.2630
1,비전공자도 이해할 수 있는 LLM 수업,박상길,비즈니스북스,과학,0.1667
2,품격 있는 대화를 위한 지식 브리핑,김진,북플레저,인문,0.1536
3,2027 공단기 심슨 독해,심우철,에스티유니타스,취업/수험서,0.0000
4,책이라면 팔 만큼 1,코지마 아오,대원씨아이,만화,0.0000


In [37]:
# 파일 존재 여부 확인을 위해 Path 불러오기
from pathlib import Path

# 추천 결과 파일이 생성되었는지 확인
print("추천 결과 파일 존재:", Path(OUTPUT_PATH).exists())

추천 결과 파일 존재: True


### 실행 결과 해석

- 추천 결과가 `chapter05_recommendations.csv` 파일로 정상적으로 저장되었다.
- 저장된 데이터의 크기는 `5행 × 5열`이다.
- 상품명, 저자, 출판사, 분야, similarity 컬럼이 모두 유지되었다.
- 저장한 CSV 파일을 다시 불러왔을 때 추천 결과의 값과 순서가 그대로 보존되었다.
- 파일 존재 여부가 `True`이므로 현재 작업 폴더에 CSV 파일이 실제로 생성되었다.
- `utf-8-sig` 인코딩을 사용하여 한글이 깨지지 않도록 저장했다.
- `index=False`를 사용하여 DataFrame의 인덱스가 별도 컬럼으로 저장되지 않았다.

### 핵심

추천 결과 DataFrame

→ `to_csv()`로 CSV 저장

→ `read_csv()`로 다시 불러오기

→ 데이터 크기와 컬럼 확인

→ 파일 존재 여부 `True`

→ 추천 결과 저장 완료

### 실습 15. 제목 기반 추천의 한계

현재 추천 시스템은 도서의 상품명만 TF-IDF 벡터로 변환하여 코사인 유사도를 계산한다.

따라서 제목에 같은 단어가 포함되면 실제 내용이나 분야가 달라도 유사도가 높게 계산될 수 있다. 반대로 같은 분야의 도서라도 제목에 공통 단어가 없으면 유사도가 `0`으로 계산될 수 있다.

이번 결과에서도 `있는`과 같은 일반적인 단어가 포함된 도서의 유사도가 높게 나타났으며, 소설·과학·인문처럼 서로 다른 분야의 도서가 함께 추천되었다.

또한 양수인 유사도를 가진 도서가 5권보다 적으면 유사도 `0`인 도서도 Top 5에 포함된다.

코사인 유사도는 현재 TF-IDF 표현에서 두 제목이 얼마나 비슷한지를 나타내는 값이다. 사용자의 취향, 도서의 품질, 구매 가능성을 의미하지 않는다.

### 핵심

유사도 높음

≠ 사용자가 반드시 좋아하는 도서

≠ 내용이나 분야가 반드시 비슷한 도서

≠ 더 좋은 도서

현재 추천 결과의 정확한 의미

→ 상품명 TF-IDF 표현을 기준으로 제목이 유사한 도서

추천 품질 개선 방법

→ 불용어 제거

→ 유사도 `0`인 도서 제외

→ 분야·저자·출판사 정보 반영

→ 사용자 클릭·평점·구매 이력 활용

### 실습 16. Streamlit 연결 준비하기

Chapter 06에서는 사용자가 웹 화면에서 도서를 선택하면 해당 도서의 인덱스를 추천 함수에 전달하도록 구성한다.

도서 선택 화면에는 인덱스와 상품명을 함께 표시하고, 실제 값으로는 도서 인덱스를 사용한다.

Streamlit 연결 흐름은 다음과 같다.

사용자 도서 선택

→ 선택한 도서의 인덱스 확인

→ `recommend_books()` 실행

→ 유사 도서 Top 5 반환

→ `st.dataframe()`으로 결과 출력

In [38]:
# Streamlit 선택 목록에 사용할 표시 문구와 도서 인덱스 연결
book_options = {
    f"{idx} | {row['상품명']}": idx
    for idx, row in df_reco.iterrows()
}

# 전체 선택 항목 개수 확인
print("도서 선택 항목 수:", len(book_options))

# 선택 목록의 앞부분 5개 확인
list(book_options.items())[:5]

도서 선택 항목 수: 989


[('0 | 세네카, 오늘을 빼앗기고 있는 당신에게', 0),
 ('1 | 홍정기의 장수근육 혁명', 1),
 ('2 | 싯다르타', 2),
 ('3 | 머니 트렌드 2027', 3),
 ('4 | 흔한남매 23', 4)]

### 실행 결과 해석

- 전체 도서 `989권`이 Streamlit의 도서 선택 항목으로 만들어졌다.
- 딕셔너리의 Key에는 `인덱스 | 상품명` 형식의 사용자 표시 문구가 저장되었다.
- 딕셔너리의 Value에는 추천 함수에 전달할 실제 도서 인덱스가 저장되었다.
- 예를 들어 사용자가 `0 | 세네카, 오늘을 빼앗기고 있는 당신에게`를 선택하면 실제 선택값은 인덱스 `0`이 된다.
- 사용자는 상품명을 보고 도서를 선택할 수 있고, 프로그램은 연결된 인덱스로 `recommend_books()`를 실행할 수 있다.

### 핵심

사용자에게 표시되는 값

→ `0 | 세네카, 오늘을 빼앗기고 있는 당신에게`

프로그램이 사용하는 실제 값

→ `0`

Streamlit 연결 흐름

→ `st.selectbox()`에서 도서 선택

→ 선택 문구를 도서 인덱스로 변환

→ `recommend_books()` 실행

→ 추천 결과를 `st.dataframe()`으로 출력

### 실습 17. 전체 추천 흐름 정리

이번 Chapter에서는 도서 제목을 TF-IDF 벡터로 변환하고, 코사인 유사도를 이용하여 제목이 비슷한 도서를 추천했다.

전체 처리 과정은 다음과 같다.

CSV 데이터 불러오기

→ 상품명 결측치와 공백 정리

→ DataFrame 인덱스 재설정

→ 도서 제목을 TF-IDF 벡터로 변환

→ 추천 기준 도서 선택

→ 전체 도서와 코사인 유사도 계산

→ 유사도 내림차순 정렬

→ 자기 자신과 동일 제목 제외

→ 유사도가 높은 Top 5 선택

→ 추천 로직을 함수로 작성

→ 추천 결과를 CSV로 저장

→ Streamlit 도서 선택 목록 준비

In [39]:
# 원본 추천 데이터와 TF-IDF 행렬의 도서 수 확인
same_row_count = (
    len(df_reco)
    == tfidf_matrix.shape[0]
)

# 추천 결과의 최대 개수 확인
valid_recommendation_count = (
    len(recommendations) <= 5
)

# 선택한 도서 자기 자신이 제외되었는지 확인
selected_title = df_reco.loc[
    selected_index,
    "상품명",
]

self_excluded = (
    selected_title
    not in recommendations["상품명"].values
)

# 유사도가 내림차순으로 정렬되었는지 확인
similarity_sorted = (
    recommendations["similarity"]
    .is_monotonic_decreasing
)

# 추천 결과 CSV 파일이 존재하는지 확인
output_file_exists = Path(
    "chapter05_recommendations.csv"
).exists()

# Streamlit 선택 항목 수가 전체 도서 수와 같은지 확인
valid_option_count = (
    len(book_options)
    == len(df_reco)
)

# 최종 검증 결과 출력
print("도서 수와 TF-IDF 행 수 일치:", same_row_count)
print("추천 결과 최대 5개:", valid_recommendation_count)
print("선택 도서 자기 자신 제외:", self_excluded)
print("유사도 내림차순 정렬:", similarity_sorted)
print("추천 결과 CSV 존재:", output_file_exists)
print("Streamlit 선택 항목 수 일치:", valid_option_count)

도서 수와 TF-IDF 행 수 일치: True
추천 결과 최대 5개: True
선택 도서 자기 자신 제외: True
유사도 내림차순 정렬: True
추천 결과 CSV 존재: True
Streamlit 선택 항목 수 일치: True


### 실행 결과 해석

- 추천 데이터의 도서 수와 TF-IDF 행렬의 행 수가 `989개`로 일치했다.
- 추천 함수가 최대 5권의 도서만 반환했다.
- 선택한 도서 자기 자신이 추천 결과에서 정상적으로 제외되었다.
- 추천 결과가 유사도 내림차순으로 정렬되었다.
- `chapter05_recommendations.csv` 파일이 정상적으로 생성되었다.
- Streamlit 선택 항목 수와 전체 도서 수가 `989개`로 일치했다.
- 모든 최종 검증 결과가 `True`이므로 추천 처리 과정이 정상적으로 완료되었다.

### 핵심

도서 제목

→ TF-IDF 벡터 변환

→ 코사인 유사도 계산

→ 유사도 내림차순 정렬

→ 자기 자신과 동일 제목 제외

→ 유사 도서 Top 5 추천

→ 추천 함수 완성

→ CSV 저장 완료

→ Streamlit 연결 준비 완료

### 최종 한 문장 정리

도서 제목을 TF-IDF 벡터로 표현하고, 선택한 도서와 전체 도서의 코사인 유사도를 계산한 뒤 자기 자신을 제외하면 제목 기반 유사 도서 추천을 만들 수 있다.

### 확인한 한계

현재 추천은 상품명만 사용하므로 제목의 공통 단어가 결과에 크게 영향을 준다. 코사인 유사도는 사용자의 취향이나 도서의 품질을 의미하지 않으며, 분야가 다른 도서나 유사도 `0`인 도서도 추천될 수 있다.